# Chapter 09: Autonomous AI Agents

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abda-dc/ai-engineering-zero-to-hero/blob/main/book/chapters/part4-agents-prod/ch09-autonomous-ai-agents.ipynb)

**Part:** Part IV: Autonomous Agents, MCP & Production MLOps  
**Chapter Identifier:** `ch09`  

## Core Concepts Covered in this Chapter:
- **Agent Anatomy**
- **Think→Act→Observe**
- **ReAct**
- **Tool/Function Calling**
- **Planning/Decomposition**
- **Short/Long-Term Memory**
- **Reflection/Self-Critique**
- **Multi-Agent Systems**
- **Orchestrator-Worker**
- **Routing**
- **LangGraph Frameworks**
- **Browser/Computer Use**
- **Code Sandboxes**
- **Human-in-the-Loop**
- **Checkpointing**
- **Cost/Latency Control**

---


## 1. Agent Anatomy & The ReAct Loop

An AI Agent is an LLM equipped with tools, memory, and a reasoning control loop. The ReAct pattern alternates between Thought (reasoning), Action (tool call), and Observation (tool result) until the objective is accomplished.

```mermaid
stateDiagram-v2
    [*] --> Thought: User Goal
    Thought --> Action: Decide Tool & Args
    Action --> Observation: Execute Tool in Sandbox
    Observation --> Reflection: Analyze Output
    Reflection --> Thought: Need More Info?
    Reflection --> FinalAnswer: Goal Met?
    FinalAnswer --> [*]
```


In [ ]:
class MockToolRegistry:
    def __init__(self):
        self.tools = {}

    def register(self, name: str, fn, schema: dict):
        self.tools[name] = {"fn": fn, "schema": schema}

    def execute(self, name: str, **kwargs):
        if name not in self.tools:
            return f"Error: Tool '{name}' not found."
        return self.tools[name]["fn"](**kwargs)

tools = MockToolRegistry()
tools.register("calculator", lambda expr: str(eval(expr, {"__builtins__": {}})), {"expr": "string"})
tools.register("weather", lambda city: f"Sunny, 22°C in {city}", {"city": "string"})

print("Weather Tool Call:", tools.execute("weather", city="San Francisco"))
print("Calculator Tool Call:", tools.execute("calculator", expr="15 * 42"))

## 2. Agent State Machines & LangGraph Architecture

Production agents avoid fragile infinite loops by modeling workflows as cyclical state graphs (e.g. LangGraph). State transitions are governed by typed state objects, conditional routing edges, and checkpointed memory.

```mermaid
flowchart LR
    A[Start] --> B[Supervisor / Router]
    B -->|Code Task| C[Coder Worker]
    B -->|Research Task| D[Search Worker]
    C --> E[Code Sandbox Execution]
    D --> F[Vector DB Lookup]
    E --> G[Synthesizer Node]
    F --> G
    G --> H{Complete?}
    H -->|No| B
    H -->|Yes| I[End]
```


In [ ]:
from typing import TypedDict, List

class AgentState(TypedDict):
    task: str
    plan: List[str]
    current_step: int
    tool_history: List[str]
    is_finished: bool

def run_agent_step(state: AgentState) -> AgentState:
    step_num = state["current_step"]
    if step_num >= len(state["plan"]):
        state["is_finished"] = True
        return state
    
    current_action = state["plan"][step_num]
    state["tool_history"].append(f"Executed: {current_action}")
    state["current_step"] += 1
    if state["current_step"] >= len(state["plan"]):
        state["is_finished"] = True
    return state

initial_state: AgentState = {
    "task": "Deploy Microservice",
    "plan": ["Run unit tests in Docker", "Apply Kubernetes manifest"],
    "current_step": 0,
    "tool_history": [],
    "is_finished": False
}

s = initial_state
while not s["is_finished"]:
    s = run_agent_step(s)
print(f"Final State Finished: {s['is_finished']}")
print(f"History: {s['tool_history']}")

## 3. Human-in-the-Loop (HITL) & Sandboxed Tool Execution

Autonomous agents must operate safely within permission boundaries. Dangerous actions (database writes, payments, file deletions) require Human-in-the-Loop authorization checkpoints.


In [ ]:
def check_action_permission(action_name: str, payload: dict) -> bool:
    DANGEROUS_ACTIONS = ["delete_file", "drop_table", "execute_payment"]
    if action_name in DANGEROUS_ACTIONS:
        print(f"[SECURITY ALERT] Action '{action_name}' requires Human-in-the-Loop authorization!")
        return False
    return True

print("Safe Read:", check_action_permission("read_file", {"path": "/etc/hosts"}))
print("Destructive Action:", check_action_permission("drop_table", {"table": "users"}))

---
## Hands-On Capstone Lab: Build an Orchestrator-Worker Multi-Agent System with Checkpointing

### Objective:
Build a multi-agent system where an Orchestrator decomposes a user request into sub-tasks, delegates to specialized Worker agents, and aggregates results.

### Instructions:
1. Define Orchestrator agent that splits task into sub-tasks.
2. Define Worker agents (ResearchWorker, CodeWorker).
3. Execute workflow and record full execution trace.


In [ ]:
class MultiAgentOrchestrator:
    def __init__(self):
        self.workers = {
            "research": lambda q: f"[Research Result] Found verified facts on: {q}",
            "code": lambda q: f"[Code Result] Generated Python script for: {q}"
        }
        self.checkpoints = []

    def execute(self, user_goal: str) -> dict:
        plan = [
            {"worker": "research", "subtask": "Analyze vector database latency benchmarks"},
            {"worker": "code", "subtask": "Write HNSW indexing script"}
        ]
        self.checkpoints.append({"step": "planning", "plan": plan})
        
        results = []
        for item in plan:
            worker_fn = self.workers[item["worker"]]
            out = worker_fn(item["subtask"])
            results.append({"worker": item["worker"], "output": out})
            self.checkpoints.append({"step": f"completed_{item['worker']}", "result": out})
            
        return {
            "goal": user_goal,
            "final_synthesis": f"Task completed successfully with {len(results)} worker contributions.",
            "trace": self.checkpoints
        }

orchestrator = MultiAgentOrchestrator()
execution = orchestrator.execute("Build vector search benchmark")
print("Synthesis:", execution["final_synthesis"])
print("Execution Checkpoints:")
for cp in execution["trace"]:
    print(f"  {cp}")

---
## Chapter Mastery Self-Check Quiz

Test your understanding of the concepts covered in this chapter.

### Question 1: What does the ReAct framework stand for in autonomous agents?
- **A)** React.js Web Framework
- **B)** Reasoning and Acting (Thought, Action, Observation cycle)
- **C)** Recursive Activation Networks
- **D)** Real-time Automated Compiler

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) Reasoning and Acting (Thought, Action, Observation cycle)

**Explanation:** ReAct combines Reasoning (generating thought traces) and Acting (calling external tools), creating an iterative problem-solving loop.
</details>

### Question 2: Why are state machine frameworks like LangGraph preferred over simple while-loops for production agents?
- **A)** They provide cyclic graphs, deterministic checkpointing, branch routing, and human-in-the-loop interruption
- **B)** They allow Python code to run in HTML
- **C)** They eliminate the need for an LLM
- **D)** They reduce token costs to zero

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** A) They provide cyclic graphs, deterministic checkpointing, branch routing, and human-in-the-loop interruption

**Explanation:** Production agents need explicit state control, persistent checkpoints to resume interrupted runs, and conditional branching, which graph abstractions provide.
</details>

### Question 3: What is the primary function of Human-in-the-Loop (HITL) design in AI agent systems?
- **A)** To force humans to write all code
- **B)** To intercept high-consequence or irreversible actions (financial, data deletion, deployment) for human approval before execution
- **C)** To slow down the agent to save electricity
- **D)** To transcribe voice audio

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) To intercept high-consequence or irreversible actions (financial, data deletion, deployment) for human approval before execution

**Explanation:** HITL establishes governance guardrails, pausing the agent's autonomous loop to require explicit human sign-off on sensitive operations.
</details>

### Question 4: In a Multi-Agent system, what is the role of an Orchestrator/Supervisor agent?
- **A)** To execute all tools directly without delegation
- **B)** To decompose complex objectives, route sub-tasks to specialized worker agents, and aggregate their outputs
- **C)** To serve static files to a web browser
- **D)** To store embeddings in a vector database

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) To decompose complex objectives, route sub-tasks to specialized worker agents, and aggregate their outputs

**Explanation:** The Orchestrator maintains high-level context, decomposes the master plan into granular subtasks, and routes them to domain-specific worker agents.
</details>
